# Estudio 06 · Paso b de clima · Variante b3 (clima completo)

Preregistro `STEP_B3` (commit `31553db`, agregado el 2026-11-06 después de ver b1 y b2; aprobado por Rosa con dos
cambios hechos antes de correr, commit `96af5cd`: en la otra regla también bloquea «compensa (empate)», y ERA5-Land se
publica con 7 días de retraso).

**b3** = L + 5a + `hum_rel_media_media5_h2` (ERA5-Land) + `precip_total_mm_media5_h2` (CHIRPS v3) +
`temp_media_media5_h2` (ERA5-Land) + `nino12_anom_media3` (la 4b de la fase 4). Lo demás, igual que L + 5a.

**Regla:** b3 se adopta si gana la alerta (`decide_multi_season`, >= 2 horizontes y ninguno «pierde» ni «compensa»)
sin «pierde» ni «compensa» en el conteo, o si gana el conteo (`decide_phase5`, misma condición) sin «pierde» ni
«compensa» en la alerta. Un empate deja L + 5a. La regla se fijó después de ver que el clima mejora en parte el
conteo en b1 y b2; b3 es la tercera comparación del paso b.

In [1]:
import json
import sys
import unittest

import numpy as np
import pandas as pd

from denguekay.config.paths import GOLD, RESEARCH
from denguekay.evaluation import episodes as diag
from denguekay.evaluation import protocol as pa
from denguekay.model import probabilistic_output as sp
from denguekay.model import tracking
from denguekay.model.device import xgboost_settings

STUDY = RESEARCH / "06_model_improvement"
sys.path.insert(0, str(STUDY / "code"))
import phase0_diagnosis as p0
import phase1_history as p1
import phase1b_prime as pb
import phase2_calibration as p2
import phase3_onsets as p3
import phase4_enso as p4
import phase5_counts as p5
import phase6_other_families as p6
import phase_b_rain as pbr
import phase_b3_full_climate as b3
import preregistration as pr

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
METRICS = STUDY / "docs" / "metricas" / "paso_b3.json"
OUTPUT_DIR = p3.OUTPUT_DIR.parent / "paso_b3"
SEED = pr.STEP_B3_COUNTERFACTUAL_SEED
device = xgboost_settings()
inp = p3.phase3_inputs()
chirps = pbr.read_chirps()
enso = p4.read_enso()
origins = pbr.add_rain(inp["origenes"], pbr.rain_by_origin_week(chirps))
origins = pb.add_humidity(origins, pb.humidity_by_origin_week(pb.era5_land_panel(inp["panel"])))
origins = b3.add_temperature(origins, b3.temperature_by_origin_week(b3.era5_land_temperature(inp["panel"])))
origins = p4.enso_at_origins(origins, enso)
columns_base, hp = p3._config()
columns = [*columns_base, *pr.STEP_B3_COLUMNS]
print(columns)
device

['hist_lag_a', 'hist_lag_b', 'hist_media_4', 'hist_semanas_pos_4', 'semana_epi_seno', 'semana_epi_coseno', 'propia_semanas_desde_ultimo_caso', 'propia_pendiente_4', 'propia_log_casos_temporada_anterior', 'hum_rel_media_media5_h2', 'precip_total_mm_media5_h2', 'temp_media_media5_h2', 'nino12_anom_media3']


{'device': 'cuda', 'n_jobs': 12}

## 1 · Guardas (antes de mirar resultados)

In [2]:
gold = pd.read_csv(GOLD / "meteo_socio_epi_piura_2017_2025_h2_gold.csv", dtype={"ubigeo": str},
                   float_precision="round_trip")
base = p6.l_plus_5a(inp["serie"])
saved5 = json.loads((STUDY / "docs" / "metricas" / "fase5.json").read_text(encoding="utf-8"))
since = pd.to_datetime(origins.origen_cierre) >= pd.Timestamp("2005-01-01")


def run_tests(names):
    suite = unittest.TestSuite(unittest.defaultTestLoader.loadTestsFromName(n) for n in names)
    result = unittest.TextTestRunner(verbosity=0).run(suite)
    return {"tests": names, "corridos": result.testsRun, "fallas": len(result.failures) + len(result.errors),
            "pasa": result.wasSuccessful()}


sys.path.insert(0, str(STUDY / "tests"))
guards = {
    "G17": pb.guard_g17(gold, inp["panel"]),
    "G19": pb.guard_g19(origins),
    "G21": p4.guard_g21(origins[origins.nino12_anom_ultimo.notna()]),
    "G22": run_tests(["test_phase4_enso.TestAvailability.test_no_leakage_by_perturbation"]),
    "G25": pbr.guard_definition(gold, inp["panel"]),
    "G26": run_tests(["test_phase_b_rain.TestDefinition.test_no_leakage_by_perturbation"]),
    "G27": pbr.guard_availability(origins, pr.STEP_B_REFERENCE_DAYS),
    "G28": pbr.guard_coverage(origins, pr.STEP_B3_COLUMNS),
    "G29": p6.guard_g16_reproduction(base["evaluaciones"], saved5["veredictos"]["5a"]["metricas"]),
    "G30": b3.guard_temperature_definition(inp["panel"]),
    "G31": run_tests(["test_phase_b3_full_climate.TestTemperature.test_no_leakage_by_perturbation"]),
    "G32": pbr.guard_coverage(origins, [pr.STEP_B3_TEMP_COLUMN]),
    "G33": b3.guard_temperature_availability(origins),
    "G34": b3.guard_raw_noaa(enso),
}
failed = [k for k, g in guards.items() if not g["pasa"]]
if failed:
    raise RuntimeError(f"Fallan guardas {failed}: se detiene b3. {guards}")
guards

----------------------------------------------------------------------
Ran 1 test in 0.043s

OK


----------------------------------------------------------------------
Ran 1 test in 0.041s

OK


----------------------------------------------------------------------
Ran 1 test in 0.035s

OK


{'G17': {'filas_gold': 30485,
  'con_valor': 30095,
  'mismos_vacios': True,
  'diferencia_maxima': 0.0,
  'pasa': True},
 'G19': {'origenes_desde_2005': 71110, 'sin_H': 0, 'pasa': True},
 'G21': {'origenes': 71175, 'con_mes_posterior': 0, 'pasa': True},
 'G22': {'tests': ['test_phase4_enso.TestAvailability.test_no_leakage_by_perturbation'],
  'corridos': 1,
  'fallas': 0,
  'pasa': True},
 'G25': {'filas_gold': 30485,
  'con_valor': 30095,
  'mismos_vacios': True,
  'diferencia_maxima': 0.0,
  'pasa': True},
 'G26': {'tests': ['test_phase_b_rain.TestDefinition.test_no_leakage_by_perturbation'],
  'corridos': 1,
  'fallas': 0,
  'pasa': True},
 'G27': {'referencia': 'origen_cierre + 14 días (hipótesis)',
  'origenes_con_lluvia': 71175,
  'sin_fecha_disponible': 0,
  'afectados': 0,
  'afectados_contra_origen_cierre': 71175,
  'margen_minimo_dias': 4,
  'pasa': True},
 'G28': {'origenes_desde_2005': 71110, 'sin_valor': 0, 'pasa': True},
 'G29': {'diferencia_maxima': 4.9451664985156185e-

## 2 · Variante b3 (5 semillas)

In [3]:
blocks = p1.seasonal_blocks(p1.FIRST_BLOCK_SEASON_LONG)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
preds, evals, fits = {}, {}, []
for seed in pr.SEEDS:
    pred = p3.predict_blocks_weighted(inp["matrices"], origins, blocks, seed, columns, hp)
    fits.append(pred.attrs.pop("ajustes").assign(semilla=seed))
    preds[seed] = pred
    evals[seed] = p5.evaluate_mixture(pred, inp["serie"])
cand = {"predicciones": preds, "evaluaciones": evals}
pd.concat(preds.values(), ignore_index=True).to_csv(OUTPUT_DIR / "predicciones_b3.csv.gz", index=False)
pd.concat(fits, ignore_index=True).drop(columns=["corte"]).to_csv(OUTPUT_DIR / "ajustes_b3.csv", index=False)

## 3 · Veredicto preregistrado (alerta y conteo frente a L + 5a)

In [4]:
alert = p3.compare(cand, base)
count = p5.compare(cand, base)
adopted = pr.adopt_step_b3(alert["veredicto"], count["veredicto"])
print("Alerta:", alert["veredicto"].veredicto.tolist(), "| Conteo:", count["veredicto"].veredicto.tolist(),
      "| se adopta b3:", adopted)
display(alert["veredicto"])
count["veredicto"]

Alerta: ['empate', 'empate', 'empate'] | Conteo: ['empate', 'empate', 'empate'] | se adopta b3: False


,horizonte,veredicto,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,a_favor_f1_050,a_favor_auprc,a_favor_brier,a_favor_razon_mae,en_contra_f1_050,en_contra_auprc,en_contra_brier,en_contra_razon_mae,a_favor_antelacion,en_contra_antelacion
0,2,empate,False,False,None,"brier, razon_mae",1,2,4,3,1,2,0,1,1,1
1,3,empate,False,False,None,"auprc, brier, razon_mae",1,3,4,3,1,2,0,1,0,2
2,4,empate,False,False,None,"auprc, brier, razon_mae",2,3,3,3,1,2,0,1,0,1


,horizonte,veredicto,vetos,vetos_contra_referencia,a_favor_razon_mae,a_favor_razon_rmse,a_favor_f1_050,a_favor_auprc,a_favor_brier,en_contra_razon_mae,en_contra_razon_rmse,en_contra_f1_050,en_contra_auprc,en_contra_brier,a_favor_antelacion,en_contra_antelacion
0,2,empate,None,brier,3,2,1,2,4,1,0,1,2,0,1,1
1,3,empate,None,"auprc, brier",3,2,1,3,4,1,1,1,2,0,0,2
2,4,empate,None,"auprc, brier",3,2,2,3,3,1,1,1,2,0,0,1


In [5]:
display(alert["antelacion"].round(3))
cols = ["horizonte", "bloque", "positivos_cand", "f1_050_cand", "f1_050_ref", "auprc_cand", "auprc_ref",
        "brier_cand", "brier_ref", "razon_mae_cand", "razon_mae_ref", "razon_rmse_cand", "razon_rmse_ref"]
alert["metricas"][cols].round(3)

,horizonte,episodios,diferencia,ic95_inferior,ic95_superior,diferencia_temporada_2010,diferencia_temporada_2015,diferencia_temporada_2016,diferencia_temporada_2022,diferencia_temporada_2023,episodios_temporada_2010,episodios_temporada_2015,episodios_temporada_2016,episodios_temporada_2022,episodios_temporada_2023
0,2,193,0.000,-0.036,0.036,0.067,0.029,-0.029,-0.053,0.000,30,34,34,38,57
1,3,193,-0.026,-0.062,0.005,0.000,0.029,-0.029,-0.053,-0.053,30,34,34,38,57
2,4,193,-0.016,-0.047,0.010,0.000,0.000,0.000,-0.053,-0.018,30,34,34,38,57


,horizonte,bloque,positivos_cand,f1_050_cand,f1_050_ref,auprc_cand,auprc_ref,brier_cand,brier_ref,razon_mae_cand,razon_mae_ref,razon_rmse_cand,razon_rmse_ref
3,2,temporada_2010,234.0,0.709,0.722,0.789,0.780,0.030,0.030,0.869,0.907,0.883,0.899
8,2,temporada_2015,557.0,0.771,0.772,0.882,0.885,0.051,0.051,0.900,0.889,0.947,0.932
9,2,temporada_2016,395.0,0.653,0.657,0.755,0.757,0.059,0.059,1.038,1.030,0.997,0.995
15,2,temporada_2022,431.0,0.700,0.712,0.781,0.781,0.053,0.054,0.888,0.946,0.886,0.965
16,2,temporada_2023,1039.0,0.861,0.857,0.949,0.947,0.061,0.062,0.788,0.808,0.813,0.893
17,2,temporada_2024,857.0,0.805,0.799,0.895,0.893,0.072,0.074,0.907,1.136,0.889,1.316
21,3,temporada_2010,234.0,0.634,0.631,0.724,0.706,0.035,0.036,0.841,0.891,0.869,0.890
26,3,temporada_2015,557.0,0.668,0.649,0.835,0.839,0.064,0.064,0.841,0.828,0.932,0.914
27,3,temporada_2016,395.0,0.631,0.627,0.705,0.709,0.064,0.065,1.046,1.042,1.021,1.016
33,3,temporada_2022,431.0,0.695,0.704,0.776,0.771,0.055,0.056,0.821,0.927,0.846,0.990


## 4 · Lecturas que no deciden

In [6]:
no2010 = alert["lectura_sin_2010"][["horizonte", "veredicto", "vetos"]]
contrast = alert["metricas"][alert["metricas"].bloque.eq(pr.block_name(pr.CONTRAST_SEASON))][cols]
display(no2010)
contrast.round(3)

,horizonte,veredicto,vetos
0,2,empate,None
1,3,empate,None
2,4,empate,None


,horizonte,bloque,positivos_cand,f1_050_cand,f1_050_ref,auprc_cand,auprc_ref,brier_cand,brier_ref,razon_mae_cand,razon_mae_ref,razon_rmse_cand,razon_rmse_ref
17,2,temporada_2024,857.0,0.805,0.799,0.895,0.893,0.072,0.074,0.907,1.136,0.889,1.316
35,3,temporada_2024,857.0,0.797,0.788,0.881,0.876,0.077,0.079,0.857,1.238,0.847,1.485
53,4,temporada_2024,857.0,0.770,0.761,0.852,0.840,0.085,0.089,0.819,1.362,0.835,1.690


In [7]:
variants = {"L + 5a": base, "b3": cand}
rd = {k: p2.readings(x, inp["serie"]) for k, x in variants.items()}
recall = pd.concat({k: x["recall_por_tipo"] for k, x in rd.items()}, names=["variante"]).reset_index(level=0)
deciding = [pr.block_name(s) for s in pr.DECIDING_SEASONS]
new = recall[recall.tipo.eq("brote_nuevo") & recall.bloque.isin(deciding)]
new_wide = new.pivot_table(index=["horizonte", "bloque"], columns="variante", values="recall")
new_wide["semanas"] = new[new.variante.eq("L + 5a")].set_index(["horizonte", "bloque"]).semanas
d = new_wide["b3"] - new_wide["L + 5a"]
print("Recall de brotes nuevos b3: mejora en", int((d > 0).sum()), "celdas, empeora en", int((d < 0).sum()), "de", len(d))
new_wide.round(3)

Recall de brotes nuevos b3: mejora en 2 celdas, empeora en 9 de 15


variante                  L + 5a     b3  semanas
horizonte bloque                                
2         temporada_2010   0.118  0.098     51.0
          temporada_2015   0.067  0.100     60.0
          temporada_2016   0.132  0.113     53.0
          temporada_2022   0.122  0.082     49.0
          temporada_2023   0.174  0.163     86.0
3         temporada_2010   0.059  0.059     51.0
          temporada_2015   0.033  0.050     60.0
          temporada_2016   0.113  0.094     53.0
          temporada_2022   0.163  0.102     49.0
          temporada_2023   0.186  0.174     86.0
4         temporada_2010   0.020  0.020     51.0
          temporada_2015   0.017  0.017     60.0
          temporada_2016   0.057  0.057     53.0
          temporada_2022   0.122  0.082     49.0
          temporada_2023   0.174  0.163     86.0

In [8]:
# Frente a la V2 en 2022 y 2023 y la HU0007-4 por celda
v2_evals = p0.evaluate_seeds(p0.v2_predictions(), inp["serie"])
med = {"V2": pa.seed_median([e["metricas"] for e in v2_evals.values()]),
       **{k: pa.seed_median([e["metricas"] for e in x["evaluaciones"].values()]) for k, x in variants.items()}}
keep = ["horizonte", "bloque", "positivos", "f1_050", "recall_050", "auprc", "brier", "razon_mae", "razon_rmse"]
v2_table = pd.concat({k: m[m.bloque.isin(["temporada_2022", "temporada_2023"])][keep] for k, m in med.items()},
                     names=["variante"]).reset_index(level=0)
hu = pd.concat({k: sp.hu0007_4_compliance(m[m.bloque.isin(["temporada_2022", "temporada_2023"])]) for k, m in med.items()},
               names=["variante"]).reset_index(level=0)
display(v2_table.round(3))
hu[["variante", "horizonte", "bloque", "cumple_clasificacion", "cumple_regresion"]]

,variante,horizonte,bloque,positivos,f1_050,recall_050,auprc,brier,razon_mae,razon_rmse
2,V2,2,temporada_2022,431.0,0.709,0.687,0.782,0.054,0.919,0.938
3,V2,2,temporada_2023,1039.0,0.858,0.821,0.948,0.063,0.818,0.869
7,V2,3,temporada_2022,431.0,0.594,0.462,0.773,0.059,0.883,0.922
8,V2,3,temporada_2023,1039.0,0.828,0.768,0.930,0.075,0.747,0.818
12,V2,4,temporada_2022,431.0,0.333,0.202,0.741,0.068,0.838,0.905
13,V2,4,temporada_2023,1039.0,0.811,0.733,0.920,0.084,0.708,0.779
15,L + 5a,2,temporada_2022,431.0,0.712,0.696,0.781,0.054,0.946,0.965
16,L + 5a,2,temporada_2023,1039.0,0.857,0.828,0.947,0.062,0.808,0.893
33,L + 5a,3,temporada_2022,431.0,0.704,0.682,0.771,0.056,0.927,0.990
34,L + 5a,3,temporada_2023,1039.0,0.835,0.794,0.930,0.073,0.763,0.852


,variante,horizonte,bloque,cumple_clasificacion,cumple_regresion
2,V2,2,temporada_2022,False,False
3,V2,2,temporada_2023,True,False
7,V2,3,temporada_2022,False,False
8,V2,3,temporada_2023,False,True
12,V2,4,temporada_2022,False,False
13,V2,4,temporada_2023,False,True
15,L + 5a,2,temporada_2022,False,False
16,L + 5a,2,temporada_2023,True,False
33,L + 5a,3,temporada_2022,False,False
34,L + 5a,3,temporada_2023,False,False


In [9]:
# b3 frente a L + 5a en 2017 y 2023
y1723 = pd.concat({k: med[k][med[k].bloque.isin(["temporada_2017", "temporada_2023"])][keep] for k in ("L + 5a", "b3")},
                  names=["variante"]).reset_index(level=0).sort_values(["bloque", "horizonte", "variante"])
y1723.round(3)

,variante,horizonte,bloque,positivos,f1_050,recall_050,auprc,brier,razon_mae,razon_rmse
10,L + 5a,2,temporada_2017,539.0,0.703,0.744,0.800,0.068,0.873,0.912
10,b3,2,temporada_2017,539.0,0.699,0.742,0.798,0.069,0.846,0.877
28,L + 5a,3,temporada_2017,539.0,0.658,0.664,0.738,0.076,0.841,0.888
28,b3,3,temporada_2017,539.0,0.657,0.664,0.739,0.076,0.832,0.871
46,L + 5a,4,temporada_2017,539.0,0.632,0.590,0.706,0.081,0.810,0.859
46,b3,4,temporada_2017,539.0,0.624,0.592,0.709,0.082,0.800,0.859
16,L + 5a,2,temporada_2023,1039.0,0.857,0.828,0.947,0.062,0.808,0.893
16,b3,2,temporada_2023,1039.0,0.861,0.837,0.949,0.061,0.788,0.813
34,L + 5a,3,temporada_2023,1039.0,0.835,0.794,0.930,0.073,0.763,0.852
34,b3,3,temporada_2023,1039.0,0.837,0.791,0.933,0.071,0.734,0.777


In [10]:
# SHAP de b3 por horizonte (semilla 17), formato de complemento_clima.shap_por_horizonte_semilla_17 del estudio 05
shap = {b: b3.shap_by_horizon(b3.fit_block(inp, origins, b, SEED, columns, hp))
        for b in ("temporada_2022", "temporada_2023", "temporada_2024")}
pd.concat(shap, names=["bloque", "horizonte"]).round(4).T

bloque                              temporada_2022                 temporada_2023                 temporada_2024                
horizonte                                      h=2     h=3     h=4            h=2     h=3     h=4            h=2     h=3     h=4
hist_lag_a                                  0.0370  0.0734  0.0997         0.0792  0.1245  0.1638         0.1005  0.1517  0.1948
hist_lag_b                                  0.0099  0.0254  0.0386         0.0278  0.0191  0.0156         0.0222  0.0160  0.0148
hist_media_4                                0.2473  0.2127  0.1753         0.2909  0.2546  0.2181         0.2563  0.2145  0.1781
hist_semanas_pos_4                          0.0150  0.0131  0.0112         0.0133  0.0116  0.0098         0.0288  0.0246  0.0207
semana_epi_seno                             0.1648  0.2030  0.2255         0.1450  0.1769  0.1963         0.1439  0.1800  0.2084
semana_epi_coseno                           0.0693  0.1179  0.1705         0.0795  0.1317  0.1859         0.0565  0.1011  0.1478
propia_semanas_desde_ultimo_caso            1.4895  1.3827  1.2799         1.4472  1.3427  1.2522         1.3374  1.2317  1.1397
propia_pendiente_4                          0.0094  0.0082  0.0070         0.0016  0.0019  0.0020         0.0252  0.0264  0.0257
propia_log_casos_temporada_anterior         0.0525  0.0636  0.0724         0.0513  0.0658  0.0765         0.0955  0.1150  0.1308
hum_rel_media_media5_h2                     0.0259  0.0391  0.0450         0.0295  0.0426  0.0492         0.0394  0.0515  0.0584
precip_total_mm_media5_h2                   0.0767  0.0926  0.0993         0.0609  0.0761  0.0823         0.1314  0.1632  0.1840
temp_media_media5_h2                        0.3395  0.4030  0.4566         0.3344  0.3945  0.4470         0.3578  0.4088  0.4499
nino12_anom_media3                          0.0431  0.0545  0.0630         0.0200  0.0261  0.0331         0.0163  0.0234  0.0329

In [11]:
# Riesgo de extrapolación del Niño: rango en los orígenes desde 2005 frente al valor actual
monthly = p4.monthly_features(enso)
current = monthly.iloc[-1]
nino_range = {**b3.nino_training_range(origins), "valor_actual": float(current.nino12_anom_media3),
              "mes_actual": current.mes_usado, "fecha_disponible_actual": str(current.fecha_disponible.date())}
nino_range

{'minimo': -1.7366666666666666,
 'maximo': 3.11,
 'origenes': 71110,
 'valor_actual': 4.673333333333333,
 'mes_actual': '2026-10',
 'fecha_disponible_actual': '2026-11-10'}

In [12]:
# Contrafactual del Niño (orígenes y valores fijados en el preregistro antes de correr)
cf = b3.nino_counterfactual(inp, origins, preds[SEED], evals[SEED]["w"], pr.STEP_B3_COUNTERFACTUAL_ORIGINS,
                            pr.STEP_B3_COUNTERFACTUAL_VALUES, columns, hp, SEED)
print("Reproducción del μ guardado con el valor real (diferencia máxima):", cf.dif_max_mu_guardada.max())
cf.drop(columns=["dif_max_mu_guardada"]).round(3)

Reproducción del μ guardado con el valor real (diferencia máxima): 0.0


,origen_cierre,bloque,horizonte,nino12_anom_media3,valor_real_medio,distritos,probabilidad_media,distritos_en_alerta,casos_predichos,casos_observados,brotes_observados
0,2017-02-04,temporada_2017,2,real,0.387,65,0.145,9,123.275,174.0,8
1,2017-02-04,temporada_2017,3,real,0.387,65,0.166,9,153.995,251.0,10
2,2017-02-04,temporada_2017,4,real,0.387,65,0.183,8,190.816,355.0,15
3,2017-02-04,temporada_2017,2,2.0,0.387,65,0.127,8,107.656,174.0,8
4,2017-02-04,temporada_2017,3,2.0,0.387,65,0.142,8,125.174,251.0,10
5,2017-02-04,temporada_2017,4,2.0,0.387,65,0.155,6,145.614,355.0,15
6,2017-02-04,temporada_2017,2,3.3,0.387,65,0.127,8,107.675,174.0,8
7,2017-02-04,temporada_2017,3,3.3,0.387,65,0.142,8,125.215,251.0,10
8,2017-02-04,temporada_2017,4,3.3,0.387,65,0.156,6,145.708,355.0,15
9,2017-02-04,temporada_2017,2,5.25,0.387,65,0.127,8,107.675,174.0,8


## 5 · Salidas: métricas y MLflow

In [13]:
to_json = lambda d: {k: diag.to_json(x) for k, x in d.items() if isinstance(x, pd.DataFrame)}
content = {
    "dispositivo": device, "guardas": guards, "veredicto_alerta": to_json(alert), "veredicto_conteo": to_json(count),
    "adoptada": bool(adopted), "base_resultante": "L + 5a + b3" if adopted else "L + 5a",
    "lectura_sin_2010": diag.to_json(no2010), "contraste_2024": diag.to_json(contrast),
    "recall_brotes_nuevos": diag.to_json(new_wide.reset_index()), "frente_a_v2": diag.to_json(v2_table),
    "hu0007_4_2022_2023": diag.to_json(hu), "temporadas_2017_2023": diag.to_json(y1723),
    "shap_por_horizonte_semilla_17": {b: diag.to_json(t.reset_index(names="horizonte")) for b, t in shap.items()},
    "rango_nino": nino_range, "contrafactual_nino": diag.to_json(cf),
}
content = json.loads(json.dumps(content, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
for section, value in content.items():
    diag.save_metrics_section(METRICS, section, value)
verdicts = pd.concat({"alerta": alert["veredicto"][["horizonte", "veredicto"]],
                      "conteo": count["veredicto"][["horizonte", "veredicto"]]}, names=["regla"]).reset_index(level=0)
run_id = tracking.log_study(
    "mejora_modelo_paso_b_lluvia_chirps", pd.DataFrame(columns=["numero", "estado"]),
    parent={"params": {"fase": "paso_b3", "preregistro": "31553db + 96af5cd", "adoptada_b3": str(bool(adopted)),
                       "base_resultante": content["base_resultante"],
                       "veredicto": "; ".join(f"{r.regla} h{r.horizonte}: {r.veredicto}" for r in verdicts.itertuples()),
                       **{f"xgb.{k}": str(x) for k, x in device.items()}},
            "metricas": {}},
    artifacts=[METRICS], tables={"veredicto": verdicts},
    description="Paso b3 del estudio 06: clima completo (humedad y temperatura ERA5-Land, lluvia CHIRPS v3, Niño 1+2) sobre L + 5a.")
run_id

C:\Users\Rosa\RufoEsEternoElRegreso\Denguekay\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow: estudio fa2a7b5bff2f45cd84d30e6d707e59b6 en 'denguekay/mejora_modelo_paso_b_lluvia_chirps' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


'fa2a7b5bff2f45cd84d30e6d707e59b6'